## Dataset exploration

I am using `bank-full.csv`. I will begin by exploring the dataset and checking for missing values, duplicate rows, and potentially invalid values.

# A description of the dataset

 Input variables:
   # bank client data:

   1 - age (numeric)

   2 - job : type of job (categorical: "admin.","unknown","unemployed","management","housemaid","entrepreneur","student",
                                       "blue-collar","self-employed","retired","technician","services")
   3 - marital : marital status (categorical: "married","divorced","single"; note: "divorced" means divorced or widowed)

   4 - education (categorical: "unknown","secondary","primary","tertiary")

   5 - default: has credit in default? (binary: "yes","no")

   6 - balance: average yearly balance, in euros (numeric)

   7 - housing: has housing loan? (binary: "yes","no")

   8 - loan: has personal loan? (binary: "yes","no")

   # Related to the last contact of the current campaign:

   9 - contact: contact communication type (categorical: "unknown","telephone","cellular")

  10 - day: last contact day of the month (numeric)

  11 - month: last contact month of year (categorical: "jan", "feb", "mar", ..., "nov", "dec")

  12 - duration: last contact duration, in seconds (numeric)

   # other attributes:
  13 - campaign: number of contacts performed during this campaign and for this client
  (numeric, includes last contact)

  14 - pdays: number of days that passed by after the client was last contacted from a
  previous campaign (numeric, -1 means client was not previously contacted)

  15 - previous: number of contacts performed before this campaign and for this client (numeric)

  16 - poutcome: outcome of the previous marketing campaign (categorical: "unknown","other","failure","success")

  Output variable (desired target):
  17 - y - has the client subscribed a term deposit? (binary: "yes","no")

8. Missing Attribute Values: None


In [62]:
import pandas as pd
bank_full_df = pd.read_csv("bank-full.csv", sep= ';')
display(bank_full_df.head(20))

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no
5,35,management,married,tertiary,no,231,yes,no,unknown,5,may,139,1,-1,0,unknown,no
6,28,management,single,tertiary,no,447,yes,yes,unknown,5,may,217,1,-1,0,unknown,no
7,42,entrepreneur,divorced,tertiary,yes,2,yes,no,unknown,5,may,380,1,-1,0,unknown,no
8,58,retired,married,primary,no,121,yes,no,unknown,5,may,50,1,-1,0,unknown,no
9,43,technician,single,secondary,no,593,yes,no,unknown,5,may,55,1,-1,0,unknown,no


In [63]:
bank_full_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y          45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 5.9 MB


The dataset contains 45,211 rows and 17 columns. The data types are `int64` and `str`.

I will examine the dataset for missing values, duplicate rows, and unusual values.

In [64]:
print("missing values by column:")
print(bank_full_df.isnull().sum()) # We have no missing values.

print("Number of identical rows:", bank_full_df.duplicated().sum()) # We have zero duplicates in this dataset

# Count "unknown" in text columns; these are categories, not NaN values.
# "unknown" is used as a category in this dataset, not as a pandas NaN value.
display(bank_full_df.select_dtypes(include=["str", "object"]).eq("unknown").sum())

missing values by column:
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64
Number of identical rows: 0


job            288
marital          0
education     1857
default          0
housing          0
loan             0
contact      13020
month            0
poutcome     36959
y                0
dtype: int64

The IQR method can flag statistically unusual values, but that does not automatically mean they are errors. For example, `pdays = -1` means that the customer was not previously contacted; it is a special code, not a missing value.

In [65]:
numerical = bank_full_df.select_dtypes(include="number")

for column in numerical.columns:
    q1 = numerical[column].quantile(0.25)
    q3 = numerical[column].quantile(0.75)
    iqr = q3 - q1

    amount = (
        (numerical[column] < q1 - 1.5 * iqr) |
        (numerical[column] > q3 + 1.5 * iqr)
    ).sum()

    print(f"{column}: {amount} possible extreme values")


age: 487 possible extreme values
balance: 4729 possible extreme values
day: 0 possible extreme values
duration: 3235 possible extreme values
campaign: 3064 possible extreme values
pdays: 8257 possible extreme values
previous: 8257 possible extreme values


In [74]:
bank_full_df.select_dtypes(include="number").max()

age             95
balance     102127
day             31
duration      4918
campaign        63
pdays          871
previous       275
dtype: int64

In [76]:
bank_full_df.select_dtypes(include="number").min()

age           18
balance    -8019
day            1
duration       0
campaign       1
pdays         -1
previous       0
dtype: int64

The observed minimum and maximum values are:

- **Age:** 18–95. The maximum is unusually high, but not necessarily incorrect.
- **Balance:** −8,019 to 102,127. Negative balances can occur.
- **Day:** 1–31, a valid range for a day of the month.
- **Duration:** 0–4,918 seconds. Call lengths vary considerably.
- **Campaign:** 1–63 contacts. The maximum is very high and worth investigating.
- **Pdays:** −1–871. A value of `-1` means the customer had not been contacted previously.
- **Previous:** 0–275 contacts. The maximum is also worth investigating.

In [77]:
display(bank_full_df.nlargest(10, "campaign"))
display(bank_full_df.nlargest(10, "previous"))
display(bank_full_df.nsmallest(10, "balance"))

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
4330,45,management,married,unknown,no,9051,yes,no,unknown,19,may,124,63,-1,0,unknown,no
11914,24,technician,single,primary,no,126,yes,no,unknown,20,jun,10,58,-1,0,unknown,no
5073,35,technician,married,secondary,no,432,yes,no,unknown,21,may,1094,55,-1,0,unknown,no
4299,30,management,single,tertiary,no,358,yes,no,unknown,19,may,88,51,-1,0,unknown,no
5459,35,blue-collar,married,secondary,no,430,yes,no,unknown,23,may,147,50,-1,0,unknown,no
18713,35,blue-collar,married,secondary,no,280,yes,yes,cellular,31,jul,65,50,-1,0,unknown,no
10697,28,blue-collar,single,secondary,no,235,yes,no,unknown,16,jun,102,46,-1,0,unknown,no
18744,49,blue-collar,married,primary,no,371,yes,no,cellular,31,jul,18,44,-1,0,unknown,no
4275,40,blue-collar,married,secondary,no,-365,yes,no,unknown,19,may,44,43,-1,0,unknown,no
13675,51,blue-collar,married,unknown,no,41,yes,no,telephone,9,jul,16,43,-1,0,unknown,no


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
29182,40,management,married,tertiary,no,543,yes,no,cellular,2,feb,349,2,262,275,other,no
38326,46,blue-collar,married,primary,no,1085,yes,yes,cellular,15,may,523,2,353,58,other,yes
44089,37,technician,married,secondary,no,432,yes,no,cellular,6,jul,386,3,776,55,failure,yes
28886,31,management,single,tertiary,no,358,yes,no,cellular,30,jan,68,3,256,51,failure,no
44822,27,blue-collar,married,secondary,no,821,yes,yes,unknown,16,sep,23,1,778,41,other,no
42611,35,technician,single,secondary,no,4645,yes,no,cellular,11,jan,502,3,270,40,other,no
28498,49,management,single,tertiary,no,145,yes,no,cellular,29,jan,57,2,248,38,failure,no
37567,39,management,married,tertiary,no,0,yes,no,cellular,14,may,11,15,261,38,failure,no
26668,51,entrepreneur,married,secondary,no,653,yes,no,cellular,20,nov,16,9,112,37,other,no
42422,27,student,single,secondary,no,91,no,no,telephone,4,dec,157,6,95,37,other,no


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
12909,26,blue-collar,single,secondary,yes,-8019,no,yes,cellular,7,jul,299,3,-1,0,unknown,no
15682,49,management,married,tertiary,yes,-6847,no,yes,cellular,21,jul,206,1,-1,0,unknown,no
38736,60,management,divorced,tertiary,no,-4057,yes,no,cellular,18,may,242,6,-1,0,unknown,no
7413,43,management,married,tertiary,yes,-3372,yes,no,unknown,29,may,386,2,-1,0,unknown,no
1896,57,self-employed,married,tertiary,yes,-3313,yes,yes,unknown,9,may,153,1,-1,0,unknown,no
32713,39,self-employed,married,tertiary,no,-3058,yes,yes,cellular,17,apr,882,3,-1,0,unknown,yes
18573,40,technician,married,tertiary,yes,-2827,yes,yes,cellular,31,jul,843,1,-1,0,unknown,no
31509,52,management,married,tertiary,no,-2712,yes,yes,cellular,2,apr,253,1,-1,0,unknown,no
25119,49,blue-collar,single,primary,yes,-2604,yes,no,cellular,18,nov,142,1,-1,0,unknown,no
14434,51,management,divorced,tertiary,no,-2282,yes,yes,cellular,14,jul,301,6,-1,0,unknown,no


The extreme-value checks flagged some unusually high or low observations. The highest campaign value is 63 contacts during the current campaign, and the highest previous value is 275 contacts before the current campaign. These values are unusual, but the other information in the rows does not clearly indicate that they are errors. The lowest balance is −8,019, which may represent a genuine negative account balance. Since these observations are unusual but not clearly invalid, I kept them in the dataset.

In [78]:
expected_categories = {
    "job": {"admin.", "unknown", "unemployed", "management", "housemaid",
            "entrepreneur", "student", "blue-collar", "self-employed",
            "retired", "technician", "services"},
    "marital": {"married", "divorced", "single"},
    "education": {"unknown", "secondary", "primary", "tertiary"},
    "default": {"yes", "no"},
    "housing": {"yes", "no"},
    "loan": {"yes", "no"},
    "contact": {"unknown", "telephone", "cellular"},
    "month": {"jan", "feb", "mar", "apr", "may", "jun",
              "jul", "aug", "sep", "oct", "nov", "dec"},
    "poutcome": {"unknown", "other", "failure", "success"},
    "y": {"yes", "no"}
}

for column, allowed in expected_categories.items():
    actual = set(bank_full_df[column].dropna().unique())
    unexpected = actual - allowed
    print(f"{column}: {sorted(unexpected) if unexpected else 'inga oväntade värden'}")

job: inga oväntade värden
marital: inga oväntade värden
education: inga oväntade värden
default: inga oväntade värden
housing: inga oväntade värden
loan: inga oväntade värden
contact: inga oväntade värden
month: inga oväntade värden
poutcome: inga oväntade värden
y: inga oväntade värden


I checked the categorical columns against the categories listed in the dataset documentation and found no unexpected values. The documented unknown values were treated as valid categories.

# Investigate the target variable `y`

In [66]:
display(bank_full_df["y"].value_counts())


y
no     39922
yes     5289
Name: count, dtype: int64

In [67]:

display((bank_full_df["y"].value_counts(normalize=True) * 100).round(1))

y
no     88.3
yes    11.7
Name: proportion, dtype: float64

### Target distribution

There are 39,922 `no` responses and 5,289 `yes` responses. Because the classes are imbalanced, a model could achieve high accuracy by frequently predicting `no` while missing many customers who actually subscribed. Recall and F1 for the `yes` class are therefore important.

# Preparing the data, training two models, and comparing their performance

In [68]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [69]:
# Exclude duration: call length is only known after the call, not before contacting the customer.
X = bank_full_df.drop(columns=["y", "duration"])
y = bank_full_df["y"]

In [70]:
numerical_columns = X.select_dtypes(include="number").columns
categorical_columns = X.select_dtypes(include=["str", "object"]).columns

In [71]:
def create_preprocessing():
    numerical = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", numerical, numerical_columns),
        ("cat", categorical, categorical_columns)
    ])


# Stratification preserves the class proportions in this random train/test split.
# The dataset is chronological, so this split does not test performance on a later time period.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "Logistic Regression": Pipeline([
        ("preprocessing", create_preprocessing()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ]),
    "Decision Tree": Pipeline([
        ("preprocessing", create_preprocessing()),
        ("model", DecisionTreeClassifier(
            max_depth=5, class_weight="balanced", random_state=42
        ))
    ])
}

for name, model in models.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)

    print(f"\n{name}")
    print("Accuracy:", round(accuracy_score(y_test, prediction), 2))
    print("Precision (yes):", round(precision_score(
        y_test, prediction, pos_label="yes", zero_division=0
    ), 2))
    print("Recall (yes):", round(recall_score(
        y_test, prediction, pos_label="yes", zero_division=0
    ), 2))
    print("F1 (yes):", round(f1_score(
        y_test, prediction, pos_label="yes", zero_division=0
    ), 2))




Logistic Regression
Accuracy: 0.75
Precision (yes): 0.27
Recall (yes): 0.62
F1 (yes): 0.37

Decision Tree
Accuracy: 0.8
Precision (yes): 0.3
Recall (yes): 0.56
F1 (yes): 0.39


The results show a trade-off between identifying as many customers who subscribed as possible and avoiding incorrect `yes` predictions:

- **Logistic regression** identifies a larger proportion of customers who actually subscribed: its recall is **0.62**, compared with **0.56** for the decision tree.
- **The decision tree** has slightly higher precision (**0.30** vs. **0.27**) and F1 (**0.39** vs. **0.37**). It performs marginally better by F1, but misses more customers who subscribed.
- Both models have low precision: only about **27–30%** of the customers predicted to subscribe actually did.

Important: about **88.3%** of customers in the dataset said “no.” A model that always predicts `no` would therefore have about **88.3% accuracy**—higher than either of your models—but it would identify no `yes` customers. This shows why accuracy alone can be misleading here. Using `class_weight="balanced"` makes the models give more weight to the less frequent class, which can improve detection of `yes` customers at the cost of overall accuracy.

 *The decision tree achieved a slightly higher F1 score for the `yes` class, while logistic regression had higher recall and therefore identified more of the customers who actually said “yes.” Both models had low precision, and their accuracy was lower than that of a simple model that always predicts `no`. Therefore, the results should be evaluated by focusing on precision, recall, and F1 for `yes`, rather than accuracy alone.*


## Summary

*The decision tree achieved a slightly higher F1 score for the `yes` class, while logistic regression had higher recall and identified more customers who subscribed. Both models had low precision. A model that always predicts `no` would have higher accuracy (about 88.3%), but it would miss every customer who subscribed. Precision, recall, and F1 for `yes` are therefore important evaluation metrics. The dataset is ordered by date, but the evaluation used a random split, so these results do not show how well the models would perform on future campaigns.*